In [ ]:
!pip install openai-whisper jiwer sacrebleu python-Levenshtein keras tensorflow torch
!pip install comet-ml unbabel-comet


In [ ]:
import whisper

# Cargar modelo (ejemplo con "base")
model_whisper = whisper.load_model("base")

# Transcribir un archivo de audio
result = model_whisper.transcribe("see-you-later-203103.wav", language="es")  # o "en"
print("Transcripción:", result["text"])


/usr/local/lib/python3.11/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcripción: 


In [ ]:
pairs = [
    ("Hello, how are you?", "Hola, ¿cómo estás?"),
    ("Buenos días, ¿cómo amaneciste?", "Good morning, how did you wake up?"),
    # ...
]


In [ ]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
# Separar datos
input_texts, target_texts = zip(*pairs)

# Tokenizar
tokenizer_in = Tokenizer()
tokenizer_in.fit_on_texts(input_texts)
X = tokenizer_in.texts_to_sequences(input_texts)

tokenizer_out = Tokenizer()
tokenizer_out.fit_on_texts(target_texts)
Y = tokenizer_out.texts_to_sequences(target_texts)

# Padding
max_len_in = max(len(seq) for seq in X)
max_len_out = max(len(seq) for seq in Y)
X = pad_sequences(X, maxlen=max_len_in, padding='post')
Y = pad_sequences(Y, maxlen=max_len_out, padding='post')

# Definir modelo Seq2Seq básico
latent_dim = 256
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(len(tokenizer_in.word_index) + 1, latent_dim)(encoder_inputs)

encoder = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder(enc_emb)
encoder_states = [state_h, state_c]

decoder_inputs = Input(shape=(None,))
dec_emb = Embedding(len(tokenizer_out.word_index) + 1, latent_dim)(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(len(tokenizer_out.word_index) + 1, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

model_seq2seq = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model_seq2seq.compile(optimizer="rmsprop", loss="sparse_categorical_crossentropy")
model_seq2seq.fit([X, Y[:, :-1]], np.expand_dims(Y[:, 1:], -1), batch_size=64, epochs=50)

# Guardar modelo
model_seq2seq.save("seq2seq_model.h5")

Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - loss: 2.3986
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step - loss: 2.3322
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - loss: 2.2681
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - loss: 2.1871
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - loss: 2.0736
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 1.9126
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step - loss: 1.7189
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - loss: 1.5682
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - loss: 1.4765
Epoch 10/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - loss: 1.4020
Epoch 11/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - loss: 1.3306
Epoch 12/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - loss: 1.2640
Epoch 13/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - loss: 1.2060
Epoch 14/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - loss: 1.1560
Epoch 15/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step - loss: 1.1110
Epoch 16/50
1/1 ━━━━━━━━━━━━━━━━

In [ ]:
from keras.models import load_model
model_seq2seq = load_model("seq2seq_model.h5")


In [ ]:
from keras.models import Model
from keras.layers import Input, LSTM, Dense, Embedding

# Supongamos que ya entrenaste tu modelo "seq2seq_model"
# y tienes:
# encoder_inputs, decoder_inputs, decoder_outputs
# además de tu tokenizer_in, tokenizer_out y max_len_in, max_len_out
from keras.models import load_model

seq2seq_model = load_model("seq2seq_model.h5")  # Ajusta el nombre
latent_dim = 256  # Debe ser el mismo usado en tu entrenamiento

# --- MODELO DE INFERENCIA DEL ENCODER ---
encoder_inputs_inf = seq2seq_model.input[0]  # Primer input del encoder
encoder_outputs, state_h_enc, state_c_enc = seq2seq_model.layers[2].output  # Capa LSTM
encoder_model = Model(encoder_inputs_inf, [state_h_enc, state_c_enc])

# --- MODELO DE INFERENCIA DEL DECODER ---
decoder_inputs_inf = seq2seq_model.input[1]
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_lstm = seq2seq_model.layers[3]  # LSTM del decoder
decoder_dense = seq2seq_model.layers[4]  # Dense final

decoder_outputs, state_h_dec, state_c_dec = decoder_lstm(
    decoder_inputs_inf, initial_state=[decoder_state_input_h, decoder_state_input_c]
)
decoder_outputs = decoder_dense(decoder_outputs)
decoder_model = Model(
    [decoder_inputs_inf, decoder_state_input_h, decoder_state_input_c],
    [decoder_outputs, state_h_dec, state_c_dec]
)


In [2]:
import sacrebleu

refs = [["Hola, ¿cómo estás?", "Buenos días"]]  # Traducciones correctas
hyps = ["Hello, how are you?", "Good morning"]

bleu = sacrebleu.corpus_bleu(hyps, [refs])
print("BLEU:", bleu.score)


ModuleNotFoundError: No module named 'sacrebleu'

In [ ]:
import Levenshtein

real = ["Buenos días", "Hola, ¿cómo estás?"]  # Transcripciones correctas
whisper_pred = ["Buenos días", "Hola, como estas"]

distancias = [Levenshtein.distance(r, p) for r, p in zip(real, whisper_pred)]
print("Promedio Levenshtein:", sum(distancias)/len(distancias))


In [ ]:
from comet.models import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = [
    {"src": "Hello, how are you?", "mt": "Hola, ¿cómo estás?", "ref": "Hola, ¿cómo estás?"},
    {"src": "Buenos días", "mt": "Good morning", "ref": "Good morning"},
]
comet_score = comet_model.predict(data, batch_size=8, gpus=1)
print("COMET score:", comet_score)


In [ ]:
try:
    from keras.preprocessing.text import Tokenizer
    from keras.preprocessing.sequence import pad_sequences
    print("Import from keras.preprocessing.text and keras.preprocessing.sequence is working.")
except ImportError:
    try:
        from tensorflow.keras.preprocessing.text import Tokenizer
        from tensorflow.keras.preprocessing.sequence import pad_sequences
        print("Import from tensorflow.keras.preprocessing.text and tensorflow.keras.preprocessing.sequence is working.")
    except ImportError:
        print("Could not import Tokenizer or pad_sequences from keras or tensorflow.keras.")